# 12.1: Batas Sebuah Pipeline Linear

Notebook ini memakai sistem RAG yang sudah Anda bangun di Sesi 11. Kita tidak
membangun apa pun yang baru di sini. Kita hanya mencari batasnya.

**Yang akan kita lakukan**

1. Menjalankan ulang pipeline Sesi 11 dalam delapan baris kode.
2. Mengajukan pertanyaan sederhana. Pipeline berhasil.
3. Mengajukan satu instruksi kompleks. Pipeline gagal.
4. Membedah mengapa gagal, dengan melihat potongan yang terambil.

Setelah notebook ini, kita akan tahu persis kemampuan apa yang hilang, dan
itulah yang dibangun di 12.2 sampai 12.4.

## Prasyarat

Susunan folder Anda harus seperti ini, supaya `rag_sesi11` menemukan index:

```
sesi-12/
├── .env                      berisi GEMINI_API_KEY
├── rag_sesi11.py
├── index_lindungiku.faiss    dari Sesi 11
├── potongan.json             dari Sesi 11
└── orchestration_limits/
    └── orchestration_limits_audience.ipynb
```

Apabila `index_lindungiku.faiss` tidak ada, salin dari folder Sesi 11 Anda

In [ ]:
import os
import sys
import textwrap

sys.path.insert(0, "..")

from dotenv import load_dotenv
from google import genai

from rag_sesi11 import cari, konteks

load_dotenv()
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

MODEL = "gemini-2.5-flash"
print("siap")

## 1. Pipeline Sesi 11, dalam delapan baris

Inilah seluruh sistem RAG Anda: ambil potongan yang relevan, tempelkan ke
prompt, minta model menjawab. Satu jalur, dari atas ke bawah, setiap kali.

Perhatikan bahwa tidak ada satu pun `if` di dalamnya.

In [ ]:
TEMPLATE = """Jawab pertanyaan HANYA berdasarkan konteks di bawah ini.
Apabila informasi tidak ditemukan, katakan bahwa Anda tidak memilikinya.

KONTEKS:
{konteks}

PERTANYAAN: {pertanyaan}"""


def jawab_linear(pertanyaan, k=3):
    blok = konteks(pertanyaan, k=k)
    prompt = TEMPLATE.format(konteks=blok, pertanyaan=pertanyaan)
    resp = client.models.generate_content(model=MODEL, contents=prompt)
    return resp.text.strip()


def tampilkan(judul, teks):
    print(judul)
    print("-" * len(judul))
    print(textwrap.fill(teks, 78))
    print()

## 2. Pertanyaan sederhana, pipeline berhasil

Satu permintaan, satu jawaban. Ini yang dirancang pipeline linear untuk lakukan,
dan ia melakukannya dengan baik.

In [ ]:
SEDERHANA = "Berapa lama masa tunggu klaim rawat inap?"

tampilkan("PERTANYAAN SEDERHANA", jawab_linear(SEDERHANA))

## 3. Instruksi kompleks, pipeline menyerah

Sekarang satu instruksi yang menuntut empat hal sekaligus:

1. mencari fakta di knowledge base,
2. membandingkannya dengan sebuah ambang,
3. melakukan hitungan,
4. lalu mengambil informasi dari dokumen yang berbeda.

Jalankan, lalu baca jawabannya dengan teliti. Bagian mana yang dikerjakan, dan
bagian mana yang dilewati?

In [ ]:
KOMPLEKS = (
    """
        Klaim rawat inap saya ditolak setelah menunggu 50 hari. Berapa masa tunggu resminya? 
        Jika saya sudah menunggu lebih lama dari itu, hitung selisihnya dalam hari, lalu jelaskan prosedur bandingnya.
    """
)

tampilkan("INSTRUKSI KOMPLEKS", jawab_linear(KOMPLEKS))

## 4. Mengapa gagal

Godaannya adalah menyalahkan modelnya. Tetapi masalahnya ada satu langkah lebih
awal, di retrieval.

Pipeline mengubah **seluruh kalimat** menjadi satu embedding. Kalimat itu berisi
tiga permintaan berbeda, jadi embedding-nya adalah semacam rata-rata dari
ketiganya, dan tidak mewakili satu pun dengan baik.

Mari kita buktikan. Bandingkan potongan yang terambil untuk instruksi kompleks
dengan potongan yang terambil kalau kita tanyakan bagiannya satu per satu.

In [ ]:
def lihat_potongan(judul, pertanyaan, k=3):
    print(judul)
    print("-" * len(judul))
    for p in cari(pertanyaan, k=k):
        print(f"  [{p['skor']:.3f}] {p['sumber']}")
    print()


lihat_potongan("Instruksi kompleks (satu embedding)", KOMPLEKS)

# TODO 1: panggil lihat_potongan dua kali lagi, masing-masing untuk satu
# bagian dari instruksi kompleks di atas. Bandingkan dokumen yang terambil.

Perhatikan apa yang terjadi. Ketika pertanyaannya dipecah, masing-masing bagian
mengambil dokumen yang tepat. Ketika digabung menjadi satu kalimat, dokumen yang
terambil bergeser, dan dokumen prosedur banding sering tidak ikut terambil sama
sekali.

Model tidak bisa menjawab tentang prosedur banding kalau dokumennya tidak pernah
sampai ke dalam prompt.

## 5. Kalau dipecah manual, ternyata bisa

Sekarang kita kerjakan secara manual apa yang nanti dikerjakan orchestration
secara otomatis: pecah instruksi, jalankan pipeline dua kali, lalu gabungkan.

Ini bukan solusi yang bagus. Kita menuliskan urutannya sendiri, jadi ia hanya
bekerja untuk instruksi yang persis seperti ini. Tetapi ia membuktikan bahwa
informasinya memang ada, dan yang kurang adalah alurnya.

In [ ]:
# TODO 2: jalankan jawab_linear dua kali, satu untuk masa tunggu dan satu
# untuk prosedur banding. Simpan hasilnya di bagian_1 dan bagian_2.
bagian_1 = None
bagian_2 = None

tampilkan("BAGIAN 1", bagian_1)
tampilkan("BAGIAN 2", bagian_2)

## 6. Satu hal yang tetap tidak bisa

Pipeline masih tidak bisa menghitung. Model menyusun teks, dan angka yang
dihasilkannya adalah tebakan yang kebetulan terlihat seperti hitungan.

Buktikan sendiri dengan hitungan yang tidak bulat.

In [ ]:
from google.genai import types

# kita coba menggunakan model gemini tanpa feature "thinking"
TANPA_THINKING = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(thinking_budget=0)
)

KLAIM = [4037000, 2353000, 4484000, 3015000, 1318000, 3276000, 238000,
         3219000, 4102000, 2395000, 3920000, 2014000, 4722000, 163000,
         1340000, 3754000, 3161000, 1479000, 2932000, 1872000]

SOAL = [
    ("Berapa 48317 dikali 29643? Jawab hanya dengan angkanya.",
     48317 * 29643),
    ("Premi tahunan 1250000 rupiah naik 8 persen setiap tahun selama 9 tahun. "
     "Berapa premi setelah kenaikan kesembilan, dibulatkan ke rupiah? "
     "Jawab hanya dengan angkanya.",
     round(1250000 * 1.08 ** 9)),
    (f"Jumlahkan nilai klaim berikut: {KLAIM}. Jawab hanya dengan angkanya.",
     sum(KLAIM)),
    ("Berapa selisih hari antara 17 Maret 2026 dan 9 Oktober 2026? "
     "Jawab hanya dengan angkanya.",
     206),
]

for soal, benar in SOAL:
    resp = client.models.generate_content(
        model=MODEL, contents=soal, config=TANPA_THINKING
    )
    print(soal[:70], "...")
    print("  jawaban model :", resp.text.strip())
    print("  jawaban benar :", benar)
    print()

## 7. Karena itu kita butuh Orchestration

Instruksi tadi berisi kata "kalau". Di dalam `jawab_linear` tidak ada satu pun
tempat untuk meletakkannya. Fungsinya selalu menjalankan langkah yang sama,
berapa pun isi jawabannya.

Kita bisa menuliskan sendiri codenya di Python (misal "if waktu tunggu > 45 hari, explain banding"), namun yang sulit adalah membuat **sistem** yang
memutuskan sendiri kapan menjalankannya, tanpa kita tuliskan satu per satu untuk
setiap kemungkinan pertanyaan.

Itulah orchestration.